# UPLIFT / MAWSIM — reproducing the AI 208 numbers

Every figure in the report comes from `docs/results/`, and every one of
those is written by a script in `scripts/` or a `--verify` in `pipeline/`.
This notebook re-runs the evaluations so a reader can check them rather
than take them.

**SIDRA is a fictional brand and its footfall is generated.** What is not
generated is the evaluation: the forecast is scored against a seasonal-naive
baseline on held-out time, and a model that cannot beat it is not used.

Run from the repository root: `uv run jupyter lab docs/artefacts/`.


In [ ]:
import sys
sys.path.insert(0, '..'); sys.path.insert(0, '../..')
from services.api.core.db import connect
conn = connect()
counts = conn.execute(
    "SELECT 'weather' k, COUNT(*) n FROM weather_hourly UNION ALL "
    "SELECT 'footfall', COUNT(*) FROM footfall_hourly UNION ALL "
    "SELECT 'events', COUNT(*) FROM events UNION ALL "
    "SELECT 'baskets', COUNT(*) FROM pos_baskets"
).fetchall()
{r[0]: r[1] for r in counts}


## 1 · The per-site argument

All four sites sit in neighbouring ERA5 grid cells and get the SAME weather.
What differs is how they respond — and the elasticity is DERIVED from the
seat counts in the brand kit, not written into the simulator, so the ordering
below is emergent rather than assumed.


In [ ]:
from services.api.brand import load_brand
from pipeline.footfall import fit_temperature_slope

for z in sorted(load_brand().zones, key=lambda z: -z.outdoor_share):
    slope = fit_temperature_slope(conn, z.code)
    print(f'{z.code}  outdoor {z.outdoor_share:6.1%}   slope {slope:+.5f} per degree')


## 2 · Forecast against the baseline

Chronological holdout, never random — a random split on an hourly series with
a weekly cycle is close to handing the model the answer.


In [ ]:
from services.api.marketing.features import load_frame
from services.api.marketing.forecast import evaluate

for zone in ('DXB-DTN', 'DXB-MOE', 'DXB-MAR', 'DXB-DEI'):
    r = evaluate(load_frame(conn, zone))
    print(f'{r.zone}  MAE {r.mae:7.2f} vs naive {r.baseline_mae:7.2f} ({r.improvement:+.1%})'
          f'   weeks {r.weeks_won}/{r.weeks_total}   cover80 {r.coverage_80:.0%}'
          f'   sMAPE {r.smape:5.1f}% vs {r.baseline_smape:5.1f}%')

# sMAPE is WORSE on every site and is reported anyway. Seasonal naive returns
# last week's exact integer, which on low-count hours is proportionally perfect
# while being absolutely further away. MAE is the gate; sMAPE is disclosed.


## 3 · Uplift, and the estimator's own bias

The bias is not noise. The donor blend is weather-flat and the treated site is
not, so the counterfactual drifts as the Gulf summer arrives. It is measured on
a placebo window where nothing happened, and subtracted.


In [ ]:
import pandas as pd
from services.api.marketing import uplift

df = pd.read_sql_query(
    "SELECT substr(ts_local,1,10) d, zone_code, SUM(footfall) f "
    "FROM footfall_hourly GROUP BY d, zone_code", conn)
daily = df.pivot(index='d', columns='zone_code', values='f').dropna().astype(float)

for inj in (0.30, 0.20, 0.10, 0.05, 0.0):
    x = uplift.recovery_check(daily, 'DXB-MAR', '2026-06-01', '2026-06-14', injected=inj)
    print(f"injected {inj:+.0%}   raw {x['recovered_raw']:+.2%}   bias {x['bias']:+.2%}"
          f"   ->  {x['recovered']:+.2%}   error {x['error_points']:+5.2f} pts"
          f"   {'ok' if x['within_5_points'] else 'FAIL'}")


## 4 · Compliance, per language

Reported per language rather than blended: the first version of the rule set
caught six violations in English, four in Arabic and two in Hindi, and one
blended number would have hidden exactly the imbalance it was written to find.


In [ ]:
from services.api.creative import compliance

for lang in ('en', 'ar', 'hi'):
    v = compliance.evaluate_by_language()[lang]
    print(f"{lang}  {v['cases']:>2} cases  "
          f"recall {v['recall']:.0%}  precision {v['precision']:.0%}  "
          f"missed {len(v['missed'])}  false alarms {len(v['false_alarms'])}")


In [ ]:
# The clean half of the gold set is deliberately adversarial: a discount WITH
# its terms, an allergen statement, 'fresh' qualified by 'baked on site'.
# Precision matters because a tool that flags good copy gets switched off.
for text in ('Our best sugar-free detox latte',
             'Was AED 32, now AED 24. Until 30 September.'):
    v = compliance.check(text)
    print(f'{text!r}\n  -> {len(v.findings)} finding(s): '
          f'{[f.rule_id for f in v.findings]}\n')


## 5 · Everything the report quotes

If any of these is missing, `scripts/build_report.py` refuses to build rather
than quietly omitting the section that cites it.


In [ ]:
from pathlib import Path

for p in sorted(Path('../results').glob('*.json')):
    print(f'{p.name:28} {p.stat().st_size:>9,} bytes')
